In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. LOAD DATA
data_path = r"C:\Users\16156\OneDrive\Desktop\Capstone Final for April\New folder\Final_all_features_combined.csv"
df = pd.read_csv(data_path) 

# 2. DEFINE FEATURES (Exact casing: Mixed Case prefixes, lowercase hjorth)
features = [
    'Normal_hjorth_complexity_q75', 
    'Normal_hjorth_complexity_mean',
    'Enter-Transition_hjorth_complexity_q75',
    'Enter-Transition_hjorth_complexity_mean',
    'Exit-Transition_hjorth_complexity_q75',
    'Exit-Transition_hjorth_complexity_mean',
    'Normal_hjorth_complexity_q50', 
    'Hypopnea_hjorth_complexity_q75',
    'Enter-Transition_hjorth_complexity_q50',
    'Exit-Transition_hjorth_complexity_q50',
    'Hypopnea_hjorth_complexity_mean', 
    'Hypopnea_hjorth_complexity_q50',
    'Normal_CNN_F50_q25', 
    'Normal_hjorth_complexity_q25',
    'Exit-Transition_hjorth_complexity_q25',
    'Enter-Transition_hjorth_complexity_q25',
    'Enter-Transition_CNN_F50_q25', 
    'Hypopnea_hjorth_complexity_q25',
    'Hypopnea_CNN_F35_std', 
    'Exit-Transition_CNN_F50_q25'
]

# 3. PRE-PROCESSING
# Mapping group to numeric for the Parallel Coordinates color scale
df['group_numeric'] = df['group'].map({'case': 1, 'control': 0})

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

# 4. ABBREVIATION MAPPING (Preserving your exact original feature names as keys)
short_labels = {
    'Normal_hjorth_complexity_q75': 'Nor hj. cmplx q75',
    'Normal_hjorth_complexity_mean': 'Nor hj. cmplx mean',
    'Enter-Transition_hjorth_complexity_q75': 'Ent hj. cmplx q75',
    'Enter-Transition_hjorth_complexity_mean': 'Ent hj. cmplx mean',
    'Exit-Transition_hjorth_complexity_q75': 'Ex hj. cmplx q75',
    'Exit-Transition_hjorth_complexity_mean': 'Ex hj. cmplx mean',
    'Normal_hjorth_complexity_q50': 'Nor hj. cmplx q50',
    'Hypopnea_hjorth_complexity_q75': 'Hyp hj. cmplx q75',
    'Enter-Transition_hjorth_complexity_q50': 'Ent hj. cmplx q50',
    'Exit-Transition_hjorth_complexity_q50': 'Ex hj. cmplx q50',
    'Hypopnea_hjorth_complexity_mean': 'Hyp hj. cmplx mean',
    'Hypopnea_hjorth_complexity_q50': 'Hyp hj. cmplx q50',
    'Normal_CNN_F50_q25': 'Nor CNN F50 q25',
    'Normal_hjorth_complexity_q25': 'Nor hj. cmplx q25',
    'Exit-Transition_hjorth_complexity_q25': 'Ex hj. cmplx q25',
    'Enter-Transition_hjorth_complexity_q25': 'Ent hj. cmplx q25',
    'Enter-Transition_CNN_F50_q25': 'Ent CNN F50 q25',
    'Hypopnea_hjorth_complexity_q25': 'Hyp hj. cmplx q25',
    'Hypopnea_CNN_F35_std': 'Hyp CNN F35 std',
    'Exit-Transition_CNN_F50_q25': 'Ex CNN F50 q25'
}

# --- VISUALIZATION 1: Parallel Coordinates ---
fig1 = px.parallel_coordinates(
    df_scaled,
    dimensions=features,
    color="group_numeric",
    color_continuous_scale=[(0, 'blue'), (1, 'red')],
    title="high-dimensional autonomic signatures: case (red) vs. control (blue)",
    labels=short_labels
)
fig1.update_layout(
    width=1800,
    height=800,
    margin=dict(t=150),
    font=dict(size=10)
)

# --- VISUALIZATION 2: PCA Scatter Plot ---
pca = PCA(n_components=2)
components = pca.fit_transform(df_scaled[features])
df_pca = pd.DataFrame(data=components, columns=['pc1', 'pc2'])
df_pca['outcome'] = df['group'] 

fig2 = px.scatter(
    df_pca, x='pc1', y='pc2', 
    color='outcome',
    title='pca: 20-feature space compression',
    color_discrete_map={'case': 'red', 'control': 'blue'}
)

# --- VISUALIZATION 3: Heatmap ---
corr_matrix = df[features].corr()
fig3 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=[short_labels[c] for c in corr_matrix.columns],
    y=[short_labels[c] for c in corr_matrix.columns],
    colorscale='RdBu',
    zmin=-1, zmax=1
))
fig3.update_layout(title='feature interaction matrix')

# --- EXPORT ---
with open('Final_Evann_Visualization_Project.html', 'w', encoding='utf-8') as f:
    f.write(fig1.to_html(full_html=True, include_plotlyjs='cdn'))
    f.write(fig2.to_html(full_html=False))
    f.write(fig3.to_html(full_html=False))

print("Dashboard complete with updated exact casing.")

Dashboard complete with updated exact casing.
